# Lab 3: Transfer Learning and CNN Interpretability

This notebook is implemented on top of Lab 2 (`CIFAR-10`) and includes:
- Model A: training from scratch (baseline from Lab 2 idea)
- Model B: pretrained model as fixed feature extractor
- Model C: full fine-tuning with different learning rates
- Interpretability: first-layer filters, activation maps, Grad-CAM, confusion matrix and error analysis


## 1. Imports and setup


In [ ]:
import os
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

RANDOM_SEED = 42
DATA_DIR = './data'
BATCH_SIZE = 64
NUM_WORKERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## 2. Data pipeline

For fairness we keep the same split indices for all models.
- Model A (from scratch): CIFAR normalization
- Models B/C (pretrained): ImageNet normalization + resize to 224 + RGB


In [ ]:
CIFAR10_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR10_STD = [0.2470, 0.2435, 0.2616]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Split indices (fixed once, reused across all transforms)
base_train = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=None)
num_train = len(base_train)
indices = np.arange(num_train)
rng = np.random.default_rng(RANDOM_SEED)
rng.shuffle(indices)

n_val = int(0.1 * num_train)
val_idx = indices[:n_val]
train_idx = indices[n_val:]

# Scratch model transforms (32x32)
transform_train_scratch = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

transform_eval_scratch = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Pretrained model transforms (224x224 + RGB + ImageNet stats)
transform_train_tl = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(224, padding=16),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_eval_tl = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Datasets/loaders for model A
train_full_scratch = datasets.CIFAR10(root=DATA_DIR, train=True, download=False, transform=transform_train_scratch)
val_full_scratch = datasets.CIFAR10(root=DATA_DIR, train=True, download=False, transform=transform_eval_scratch)
test_scratch = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform_eval_scratch)

train_scratch = Subset(train_full_scratch, train_idx)
val_scratch = Subset(val_full_scratch, val_idx)

train_loader_scratch = DataLoader(train_scratch, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader_scratch = DataLoader(val_scratch, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader_scratch = DataLoader(test_scratch, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Datasets/loaders for models B/C
train_full_tl = datasets.CIFAR10(root=DATA_DIR, train=True, download=False, transform=transform_train_tl)
val_full_tl = datasets.CIFAR10(root=DATA_DIR, train=True, download=False, transform=transform_eval_tl)
test_tl = datasets.CIFAR10(root=DATA_DIR, train=False, download=False, transform=transform_eval_tl)

train_tl = Subset(train_full_tl, train_idx)
val_tl = Subset(val_full_tl, val_idx)

train_loader_tl = DataLoader(train_tl, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader_tl = DataLoader(val_tl, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader_tl = DataLoader(test_tl, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

class_names = base_train.classes
num_classes = len(class_names)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_tl)}')
print('Classes:', class_names)


## 3. Training utilities


In [ ]:
def count_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X.size(0)
    return running_loss / len(loader.dataset)


def evaluate_accuracy(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total if total else 0.0


def train_model(model, train_loader, val_loader, optimizer, device, max_epochs=20, patience=5, scheduler=None):
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_acc': [], 'epoch_time_sec': []}

    best_val = 0.0
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        t0 = time.perf_counter()
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_acc = evaluate_accuracy(model, val_loader, device)
        dt = time.perf_counter() - t0

        if scheduler is not None:
            scheduler.step(val_acc)

        history['train_loss'].append(train_loss)
        history['val_acc'].append(val_acc)
        history['epoch_time_sec'].append(dt)

        print(f'Epoch {epoch+1:02d}/{max_epochs} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f} | time={dt:.1f}s')

        if val_acc > best_val:
            best_val = val_acc
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return history


def plot_history(history, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='train loss')
    ax1.set_xlabel('Epoch')
    ax1.set_title(f'Train Loss {title}')
    ax1.legend()

    ax2.plot(history['val_acc'], label='val accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_title(f'Val Accuracy {title}')
    ax2.legend()

    plt.tight_layout()
    plt.show()


## 4. Model A (from scratch, based on Lab 2)


In [ ]:
class CNNImproved(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Dropout2d(0.3),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model_a = CNNImproved(num_classes=num_classes).to(device)
opt_a = torch.optim.AdamW(model_a.parameters(), lr=1e-3, weight_decay=1e-4)
sch_a = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_a, mode='max', factor=0.5, patience=2)

history_a = train_model(model_a, train_loader_scratch, val_loader_scratch, opt_a, device, max_epochs=25, patience=5, scheduler=sch_a)
plot_history(history_a, '(Model A)')

val_acc_a = max(history_a['val_acc'])
test_acc_a = evaluate_accuracy(model_a, test_loader_scratch, device)
print(f'Model A | best val={val_acc_a:.4f}, test={test_acc_a:.4f}')


## 5. Model B (pretrained + frozen backbone)


In [ ]:
weights = models.ResNet18_Weights.DEFAULT

model_b = models.resnet18(weights=weights)
for p in model_b.parameters():
    p.requires_grad = False

in_features = model_b.fc.in_features
model_b.fc = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes),
)
model_b = model_b.to(device)

opt_b = torch.optim.Adam(filter(lambda p: p.requires_grad, model_b.parameters()), lr=1e-3)
sch_b = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_b, mode='max', factor=0.5, patience=2)

history_b = train_model(model_b, train_loader_tl, val_loader_tl, opt_b, device, max_epochs=15, patience=4, scheduler=sch_b)
plot_history(history_b, '(Model B)')

val_acc_b = max(history_b['val_acc'])
test_acc_b = evaluate_accuracy(model_b, test_loader_tl, device)
print(f'Model B | best val={val_acc_b:.4f}, test={test_acc_b:.4f}')


## 6. Model C (pretrained + full fine-tuning)


In [ ]:
model_c = models.resnet18(weights=weights)
in_features = model_c.fc.in_features
model_c.fc = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes),
)
model_c = model_c.to(device)

head_params = list(model_c.fc.parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in model_c.parameters() if id(p) not in head_ids]

opt_c = torch.optim.Adam([
    {'params': backbone_params, 'lr': 1e-4},
    {'params': head_params, 'lr': 1e-3},
])
sch_c = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_c, mode='max', factor=0.5, patience=2)

history_c = train_model(model_c, train_loader_tl, val_loader_tl, opt_c, device, max_epochs=15, patience=4, scheduler=sch_c)
plot_history(history_c, '(Model C)')

val_acc_c = max(history_c['val_acc'])
test_acc_c = evaluate_accuracy(model_c, test_loader_tl, device)
print(f'Model C | best val={val_acc_c:.4f}, test={test_acc_c:.4f}')


## 7. Comparison table for models A/B/C


In [ ]:
def convergence_epoch(history):
    vals = history['val_acc']
    return int(np.argmax(vals) + 1)

rows = []
for name, mode, model, history, val_acc, test_acc in [
    ('A', 'From scratch', model_a, history_a, val_acc_a, test_acc_a),
    ('B', 'Pretrained + frozen', model_b, history_b, val_acc_b, test_acc_b),
    ('C', 'Fine-tuning', model_c, history_c, val_acc_c, test_acc_c),
]:
    trainable, total = count_params(model)
    rows.append({
        'Model': name,
        'Mode': mode,
        'Trainable params': trainable,
        'Total params': total,
        'Time/epoch (sec)': float(np.mean(history['epoch_time_sec'])),
        'Epoch to best val': convergence_epoch(history),
        'Best val acc (%)': val_acc * 100,
        'Test acc (%)': test_acc * 100,
    })

df_compare = pd.DataFrame(rows)
df_compare


## 8. Select the best model for interpretability


In [ ]:
model_pool = {
    'A': {'model': model_a, 'test_acc': test_acc_a, 'loader': test_loader_scratch, 'norm_mean': CIFAR10_MEAN, 'norm_std': CIFAR10_STD, 'input_size': 32},
    'B': {'model': model_b, 'test_acc': test_acc_b, 'loader': test_loader_tl, 'norm_mean': IMAGENET_MEAN, 'norm_std': IMAGENET_STD, 'input_size': 224},
    'C': {'model': model_c, 'test_acc': test_acc_c, 'loader': test_loader_tl, 'norm_mean': IMAGENET_MEAN, 'norm_std': IMAGENET_STD, 'input_size': 224},
}

best_name = max(model_pool.keys(), key=lambda k: model_pool[k]['test_acc'])
best_cfg = model_pool[best_name]
best_model = best_cfg['model']
best_loader = best_cfg['loader']
BEST_MEAN = best_cfg['norm_mean']
BEST_STD = best_cfg['norm_std']

print(f'Best model: {best_name}, test_acc={best_cfg["test_acc"]:.4f}')


## 9. Interpretability Part A: first conv filters


In [ ]:
def find_first_conv(model):
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            return module
    return None

first_conv = find_first_conv(best_model)
assert first_conv is not None, 'No Conv2d layer found'

filters = first_conv.weight.detach().cpu()
fmin, fmax = filters.min(), filters.max()
filters_norm = (filters - fmin) / (fmax - fmin + 1e-8)

n_show = min(16, filters_norm.shape[0])
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis('off')
        continue
    f = filters_norm[i]
    if f.shape[0] == 1:
        ax.imshow(f[0], cmap='gray')
    else:
        ax.imshow(f.permute(1, 2, 0).clamp(0, 1))
    ax.axis('off')
plt.suptitle('First conv filters (normalized)')
plt.tight_layout()
plt.show()


## 10. Interpretability Part B: activation maps


In [ ]:
def denormalize(img_t, mean, std):
    t = img_t.detach().cpu().clone()
    mean_t = torch.tensor(mean).view(-1, 1, 1)
    std_t = torch.tensor(std).view(-1, 1, 1)
    return (t * std_t + mean_t).clamp(0, 1)


def collect_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels, all_imgs = [], [], []
    with torch.no_grad():
        for X, y in loader:
            logits = model(X.to(device))
            preds = logits.argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(y.cpu())
            all_imgs.append(X.cpu())
    return torch.cat(all_imgs), torch.cat(all_labels), torch.cat(all_preds)

all_imgs, all_labels, all_preds = collect_predictions(best_model, best_loader, device)
correct_idx = (all_preds == all_labels).nonzero(as_tuple=True)[0]
wrong_idx = (all_preds != all_labels).nonzero(as_tuple=True)[0]

sample_correct = correct_idx[0].item() if len(correct_idx) > 0 else 0
sample_wrong = wrong_idx[0].item() if len(wrong_idx) > 0 else 0
sample_ids = [sample_correct, sample_wrong]

# Pick early and deep conv layers for hooks
conv_layers = [m for m in best_model.modules() if isinstance(m, nn.Conv2d)]
assert len(conv_layers) >= 2, 'Need at least two conv layers'
early_layer = conv_layers[0]
deep_layer = conv_layers[-1]

activations = {}
handles = []

for name, layer in [('early', early_layer), ('deep', deep_layer)]:
    def _make_hook(key):
        def hook(module, inp, out):
            activations[key] = out.detach().cpu()
        return hook
    handles.append(layer.register_forward_hook(_make_hook(name)))

best_model.eval()
for idx in sample_ids:
    x = all_imgs[idx:idx+1].to(device)
    y_true = all_labels[idx].item()
    y_pred = all_preds[idx].item()
    with torch.no_grad():
        _ = best_model(x)

    img_show = denormalize(all_imgs[idx], BEST_MEAN, BEST_STD).permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(3, 4, figsize=(12, 8))
    axes[0, 0].imshow(img_show)
    axes[0, 0].set_title(f'Input\ntrue={class_names[y_true]}\npred={class_names[y_pred]}')
    axes[0, 0].axis('off')

    for j in range(1, 4):
        axes[0, j].axis('off')

    early_maps = activations['early'][0]
    deep_maps = activations['deep'][0]

    for j in range(4):
        axes[1, j].imshow(early_maps[j].numpy(), cmap='gray')
        axes[1, j].set_title(f'Early ch {j}')
        axes[1, j].axis('off')

    for j in range(4):
        axes[2, j].imshow(deep_maps[j].numpy(), cmap='gray')
        axes[2, j].set_title(f'Deep ch {j}')
        axes[2, j].axis('off')

    plt.tight_layout()
    plt.show()

for h in handles:
    h.remove()


## 11. Interpretability Part C: Grad-CAM


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, x, class_idx=None):
        self.model.eval()
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        self.model.zero_grad()
        logits[0, class_idx].backward(retain_graph=True)

        grads = self.gradients
        acts = self.activations
        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    def close(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()


def find_last_conv(model):
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    return convs[-1] if convs else None


target_layer = find_last_conv(best_model)
assert target_layer is not None, 'No Conv2d layer found for Grad-CAM'

gradcam = GradCAM(best_model, target_layer)


In [ ]:
# Select at least 8 samples: 4 correct (preferably from different classes) + 4 wrong
correct_idxs = (all_preds == all_labels).nonzero(as_tuple=True)[0].tolist()
wrong_idxs = (all_preds != all_labels).nonzero(as_tuple=True)[0].tolist()

selected_correct = []
used_classes = set()
for idx in correct_idxs:
    cls = int(all_labels[idx])
    if cls not in used_classes:
        selected_correct.append(idx)
        used_classes.add(cls)
    if len(selected_correct) == 4:
        break

selected_wrong = wrong_idxs[:4]
selected = selected_correct + selected_wrong

print('Selected correct:', len(selected_correct), '| selected wrong:', len(selected_wrong))

ncols = 4
nrows = int(np.ceil(len(selected) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.array(axes).reshape(-1)

for i, idx in enumerate(selected):
    x = all_imgs[idx:idx+1].to(device)
    y_true = int(all_labels[idx])
    y_pred = int(all_preds[idx])

    cam, pred_idx = gradcam.generate(x)
    img = denormalize(all_imgs[idx], BEST_MEAN, BEST_STD).permute(1, 2, 0).numpy()

    ax = axes[i]
    ax.imshow(img)
    ax.imshow(cam, cmap='jet', alpha=0.45)
    mark = 'OK' if y_true == y_pred else 'ERR'
    ax.set_title(f'{mark} | t={class_names[y_true]} | p={class_names[pred_idx]}')
    ax.axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


## 12. Interpretability Part D: confusion matrix and top errors


In [ ]:
cm = confusion_matrix(all_labels.numpy(), all_preds.numpy())

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.title('Confusion Matrix (best model)')
plt.show()

# top-3 confusion pairs (true -> predicted), excluding diagonal
pairs = []
for t in range(cm.shape[0]):
    for p in range(cm.shape[1]):
        if t != p and cm[t, p] > 0:
            pairs.append((cm[t, p], t, p))

pairs_sorted = sorted(pairs, reverse=True)[:3]
print('Top-3 confusion pairs:')
for cnt, t, p in pairs_sorted:
    print(f'{class_names[t]} -> {class_names[p]}: {cnt}')


In [ ]:
# Show 2 examples for each top confusion pair with Grad-CAM
all_labels_np = all_labels.numpy()
all_preds_np = all_preds.numpy()

for cnt, t, p in pairs_sorted:
    idxs = np.where((all_labels_np == t) & (all_preds_np == p))[0][:2]
    if len(idxs) == 0:
        continue

    fig, axes = plt.subplots(1, len(idxs), figsize=(5 * len(idxs), 4))
    if len(idxs) == 1:
        axes = [axes]

    for ax, idx in zip(axes, idxs):
        x = all_imgs[idx:idx+1].to(device)
        cam, pred_idx = gradcam.generate(x)
        img = denormalize(all_imgs[idx], BEST_MEAN, BEST_STD).permute(1, 2, 0).numpy()

        ax.imshow(img)
        ax.imshow(cam, cmap='jet', alpha=0.45)
        ax.set_title(f'true={class_names[t]}\npred={class_names[pred_idx]}')
        ax.axis('off')

    plt.suptitle(f'Confusion pair: {class_names[t]} -> {class_names[p]} (count={cnt})')
    plt.tight_layout()
    plt.show()


In [ ]:
gradcam.close()


## 13. Short conclusions template

Fill this section after running the notebook:

1. Which model is best on test set and why?
2. Compare A/B/C by trainable parameters and training speed.
3. What do first-layer filters look like (edges, color transitions, etc.)?
4. What is visible in early vs deep activation maps?
5. Do Grad-CAM maps focus on semantically meaningful object regions?
6. Top confusion pairs and possible reasons for errors.
